In [1]:
import pandas as pd

# 1. Load the file without the memory warning
df = pd.read_csv("mails_data.csv", low_memory=False)

# Let's see what the mess looks like
print(f"Original shape: {df.shape}")

Original shape: (103534, 422)


In [2]:
# 2. Look at your column names to confirm what the text column is called
print(df.columns)

# 3. Assuming your columns are named 'label' and 'text', keep ONLY those two
# (If your text column is named something else like 'Message', change it here)
df = df[['label', 'text']]

print(f"Shape after dropping unnamed columns: {df.shape}")

Index(['label', 'text', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       ...
       'Unnamed: 412', 'Unnamed: 413', 'Unnamed: 414', 'Unnamed: 415',
       'Unnamed: 416', 'Unnamed: 417', 'Unnamed: 418', 'Unnamed: 419',
       'Unnamed: 420', 'Unnamed: 421'],
      dtype='str', length=422)
Shape after dropping unnamed columns: (103534, 2)


In [3]:
# 4. Drop any rows where the 'text' is missing (NaN)
df = df.dropna(subset=['text'])

# 5. Reset the index so your row numbers are clean
df.reset_index(drop=True, inplace=True)

print(f"Final clean shape: {df.shape}")

Final clean shape: (93976, 2)


In [4]:
# Check the initial shape
print(f"Initial shape: {df.shape}")

# Drop exact duplicate emails
df = df.drop_duplicates(subset=['text'])

# Drop any rows where the text might be missing (NaN)
df = df.dropna(subset=['text'])

print(f"Shape after cleaning: {df.shape}")

Initial shape: (93976, 2)
Shape after cleaning: (92080, 2)


In [5]:
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove punctuations and numbers (keep only letters)
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = text.strip()
    return text

# Apply the cleaning function to create a new cleaned column
df['cleaned_text'] = df['text'].apply(clean_text)

In [8]:
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer (you can also remove english stop words here)
vectorizer = TfidfVectorizer(stop_words='english', max_features=2500)

# Fit the vectorizer on the cleaned text and transform it into a matrix
X = vectorizer.fit_transform(df['cleaned_text'])

# The target labels for XGBoost
y = df['label']

print(f"Matrix shape: {X.shape}") 
# X is now your numerical feature matrix ready for XGBoost

Matrix shape: (92080, 2500)
